# 01. Source to bytecode

**Tier:** Deep. **Why you'd use this:** You want to see what `compile` produces and why loading a local is cheaper than a global. Bytecode makes that cost visible.

**After this lesson you can:** say what `compile` returns; read which names are locals; explain why a local is cheaper to load than a global.
About 30 minutes. **Uses:** [The import system](../09-imports/01-import-system-lesson.ipynb), [Scopes and closures](../06-scopes/01-closures-lesson.ipynb)


## Quick review

1. Where does Python cache an imported module?
2. Does `importlib.reload` update a name you already bound to an old object from that module?

<details><summary>Answers</summary>

`sys.modules`. No — your name still points at the old object unless you bind it again.
</details>


## The idea

Python does not walk your source text on every call. It compiles the text to a code object, a bundle of bytecode plus tables of names and constants. A function call runs that bytecode in a frame. Locals are slots in the frame, addressed by number. Globals are a dictionary, addressed by name.

Import saves the code object in a `__pycache__` file ending in `.pyc` so the next process can skip the compile when the source has not changed.

<details><summary>Explain more</summary>

`dis.dis` prints the bytecode. The exact instruction names are a CPython detail and can change. What stays true is the split: a local load uses the frame array, a global load searches a dictionary.

`code.co_varnames` is the tuple of local names. `code.co_names` is the tuple of global and attribute names. Those tables are how the numbered operands find a name when a human reads the disassembly.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

This is the bytecode a virtual machine runs, closer to JVM bytecode than to a native binary. `.pyc` is the cached class file. A local is a slot. A global is a hash-table lookup, the same idea as a static field resolved by name.
</details>


## See it work

A tight loop adds a base amount on every pass. Reading `base` from the module on every pass searches the global dictionary each time the instruction runs. Copying it into a local once makes the loop body load a slot. The results match. The bytecode does not.


In [1]:
def with_global(n: int) -> int:
    total = 0
    for _ in range(n):
        total = total + base
    return total


def with_local(n: int) -> int:
    # 1. One global read, stored in a frame slot.
    local_base = base
    total = 0
    for _ in range(n):
        # 2. The loop body loads the slot, not the module dict.
        total = total + local_base
    return total


base = 3
print(with_global(4), with_local(4))
print("base" in with_global.__code__.co_names)
print("local_base" in with_local.__code__.co_varnames)
print("total" in with_local.__code__.co_varnames)

12 12
True
True
True


| function | `base` lives in | loop body loads |
| --- | --- | --- |
| `with_global` | `co_names` | the global, every pass |
| `with_local` | a local after one copy | `local_base`, a slot |

```
frame slots: [n, local_base, total, _]
global dict: {"base": 3, "with_global": ...}
```

Both calls return 12. The equality of the answers is why the slower load is easy to miss.


## Check yourself

Answer these from the example above before you run the cell.


1. Why does `compile` not set `answer` in the notebook?
2. Why is `local_base` absent from `co_names` of the local version's loop body even though `base` may still appear once?
3. What is a `.pyc` file a cache of?

<details><summary>Answers</summary>

1. `compile` builds a code object. Nothing executes it. `exec(code)` would perform the store. The example above never calls `exec`.
2. The copy `local_base = base` is one global load, before the loop. The name `base` can still sit in `co_names` for that one instruction. The additions inside the loop name `local_base`, which is a varname, so those instructions do not search the global dict.
3. The code object produced from the source. It is not a copy of the module's data, and it is not consulted if the source looks newer than the cache.
</details>


In [2]:
code = compile("answer = 1 + 2\n", "<lesson>", "exec")
print(type(code).__name__)
print(code.co_names)

code
('answer',)


<details><summary>Check your answers</summary>

- The type name is `code`. `compile` did not run the assignment. It returned the object the interpreter would run.
- `co_names` is `('answer',)`. That is the global name the bytecode stores into. The sum itself is not a name.
</details>


## Why it works this way

`compile` parses the text, checks the grammar, and emits a code object. Running it is a separate step, `exec` or a function call. That split is why a syntax error happens before any line runs, and why `compile` in the example above prints a type name instead of assigning `answer`. The code object holds the bytecode, `co_consts` for constants the instructions refer to, `co_names` for globals and attributes, and `co_varnames` for locals. A function's `__code__` is that object.

A local load does not hash a string. The compiler assigned `total` an index. The instruction carries that index. The frame is an array, and the load is one indexed read. A global load carries an index into `co_names`, then looks that string up in the global dictionary and, on a miss, in builtins. A dictionary lookup is more work than an array read, and it happens every time the instruction runs. A loop that loads a global once per pass repeats that lookup. Copying into a local moves the lookup to one instruction before the loop. The body then uses the fast index. The timeit cell measures that gap. The numbers move from run to run, so the cell is volatile. The direction is the point: the local version finishes first when the loop is long enough for the loads to matter.

The disassembly names the instructions. On this build a local load is in the `LOAD_FAST` family, often printed `LOAD_FAST_BORROW`, and a local store is `STORE_FAST`. A global load is `LOAD_GLOBAL` and a global store is `STORE_GLOBAL`. CPython detail: those spellings change. `LOAD_FAST_BORROW` is a newer way to read a local without an extra reference-count bump when the interpreter can prove the value stays alive. The next Python can print a different name for the same idea. The volatile `dis` cell is a picture of this build, not a contract. The stable contract is `co_varnames` versus `co_names`.

Import compiles if it must, then writes the code object to `__pycache__/*.pyc`. The next process checks whether that file still matches the source. If it does, import loads the code object and skips the parser. That is why a repeated import is cheap even the first time in a new process, after the cache exists. The check can be fooled when a bytecode file looks newer than a source edit made in the same second, which is why the import lesson deleted `__pycache__` before `reload`. The pyc is a cache of the compile, not a second copy of the module's data.

The design reason for bytecode is a portable, inspectable form that is faster to run than walking a syntax tree, without committing to one CPU. The design reason for the local array is the scope rule: the compiler already knows every local, so it can number them. Globals are open-ended, so they stay a dictionary. You pay for that openness on every load.


In [3]:
import dis


def trace_global(n: int) -> int:
    total = 0
    for _ in range(n):
        total = total + base
    return total


def trace_local(n: int) -> int:
    local_base = base
    total = 0
    for _ in range(n):
        total = total + local_base
    return total


dis.dis(trace_global)
print("---")
dis.dis(trace_local)

  4           RESUME                   0

  5           LOAD_SMALL_INT           0
              STORE_FAST               1 (total)

  6           LOAD_GLOBAL              1 (range + NULL)
              LOAD_FAST_BORROW         0 (n)
              CALL                     1
              GET_ITER
      L1:     FOR_ITER                16 (to L2)
              STORE_FAST               2 (_)

  7           LOAD_FAST_BORROW         1 (total)
              LOAD_GLOBAL              2 (base)
              BINARY_OP                0 (+)
              STORE_FAST               1 (total)
              JUMP_BACKWARD           18 (to L1)

  6   L2:     END_FOR
              POP_ITER

  8           LOAD_FAST_BORROW         1 (total)
              RETURN_VALUE
---
 11           RESUME                   0

 12           LOAD_GLOBAL              0 (base)
              STORE_FAST               1 (local_base)

 13           LOAD_SMALL_INT           0
              STORE_FAST               2 (total)

 14 

In [4]:
import timeit


def bench_global() -> int:
    total = 0
    for _ in range(2000):
        total = total + base
    return total


def bench_local() -> int:
    local_base = base
    total = 0
    for _ in range(2000):
        total = total + local_base
    return total


print(round(timeit.timeit(bench_global, number=100), 4))
print(round(timeit.timeit(bench_local, number=100), 4))

0.004
0.0035


In [5]:
import importlib
import importlib.util
import sys
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())
sys.path.insert(0, str(folder))
source = folder / "tiny.py"
source.write_text("value = 1\n")
importlib.invalidate_caches()
tiny = importlib.import_module("tiny")
cached = importlib.util.cache_from_source(str(source))
print(cached.endswith(".pyc"))
print("__pycache__" in cached)

True
True


## When to use it

Copy a global into a local when a loop would otherwise look the name up on every pass and a profile says that loop matters. Do it for a function, a method, or a module constant you read thousands of times. Do not do it for a single read. The assignment is noise if the lookup is not in the hot path.

Read `dis` when you want to know whether a name is local. Do not write code that depends on an opcode name or an opcode number. Those are not a public API. Depend on `co_varnames` and `co_names` if you are inspecting a function, and even that is for understanding, not for a program's logic.

Leave `__pycache__` alone in normal work. Delete it when an experiment rewrites a file and reload seems to ignore the edit. Commit source, not `.pyc` files.


## Common mistake

A counter in the module, updated with `global`, is one object every call continues. Two calls return 3 and then 6. The function looks correct if you call it once. The bytecode stores a global, so the second call sees the first call's writes.


In [6]:
total = 0


def add_three() -> int:
    global total
    for _ in range(3):
        total = total + 1
    return total


print(add_three())
print(add_three())
print("total" in add_three.__code__.co_names)

3
6
True


The prints are `3`, `6`, and `True`. `total` is in `co_names`, not in the frame's local array, so each call resumes the module's integer. The fix drops `global` and uses a local. Each call then returns 3, and `total` is a varname.


In [7]:
def add_three_local() -> int:
    total = 0
    for _ in range(3):
        total = total + 1
    return total


print(add_three_local())
print(add_three_local())
print("total" in add_three_local.__code__.co_varnames)

3
3
True


## Try it

Take `range` itself into a local before the loop in `with_global`, the same way `local_base` copies `base`. Which table, `co_names` or `co_varnames`, gains the new local, and which load inside the loop disappears from a fresh `dis` listing?


## What this explains

- [The import system](../09-imports/01-import-system-lesson.ipynb): import compiles when the `.pyc` is missing or stale, then runs the code object. The cache is why a second process can skip the parser.
- [Scopes and closures](../06-scopes/01-closures-lesson.ipynb): a name assigned in the function is a local slot. The compiler numbers those slots. That numbering is `co_varnames` and the fast load.
- [How list works](../03-lists/01-dynamic-array-lesson.ipynb): the frame's locals are a small array, the same idea as indexed access being cheaper than a search.


## Recap

- Source compiles to a code object. `co_varnames` are locals. `co_names` are globals and attributes.
- A local load is an index into the frame. A global load searches a dictionary, and a loop repeats that search.
- A `.pyc` caches the code object. Opcode names in `dis` are a picture of this build.

Next: [Memory management](../11-memory/01-refcount-lesson.ipynb).
